## Quick introduction to generating counterfactual explanations using DiCE
https://interpret.ml/DiCE/notebooks/DiCE_getting_started.html

In [55]:
# Sklearn imports
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# DiCE imports
import dice_ml
from dice_ml.utils import helpers  # helper functions

# pandas
import pandas as pd

In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
# Preliminaries: Loading a dataset and a ML model trained over it
dataset = helpers.load_adult_income_dataset()
dataset.head()
# This dataset has 8 features.The outcome is income which is binarized to 0 (low-income, <=50K) or 1 (high-income, >50K).

/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/dice_ml/utils/helpers.py:79: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adult_data = adult_data.replace({'income': {'<=50K': 0, '>50K': 1}})
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: F

,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,28,Private,Bachelors,Single,White-Collar,White,Female,60,0
1,30,Self-Employed,Assoc,Married,Professional,White,Male,65,1
2,32,Private,Some-college,Married,White-Collar,White,Male,50,0
3,20,Private,Some-college,Single,Service,White,Female,35,0
4,41,Self-Employed,Some-college,Married,White-Collar,White,Male,50,0


In [58]:
# description of transformed features
adult_info = helpers.get_adult_data_info()
adult_info

{'age': 'age',
 'workclass': 'type of industry (Government, Other/Unknown, Private, Self-Employed)',
 'education': 'education level (Assoc, Bachelors, Doctorate, HS-grad, Masters, Prof-school, School, Some-college)',
 'marital_status': 'marital status (Divorced, Married, Separated, Single, Widowed)',
 'occupation': 'occupation (Blue-Collar, Other/Unknown, Professional, Sales, Service, White-Collar)',
 'race': 'white or other race?',
 'gender': 'male or female?',
 'hours_per_week': 'total work hours per week',
 'income': '0 (<=50K) vs 1 (>50K)'}

In [59]:
# Split the dataset into train and test sets.
target = dataset["income"]
train_dataset, test_dataset, y_train, y_test = train_test_split(dataset,
                                                                target,
                                                                test_size=0.2,
                                                                random_state=0,
                                                                stratify=target)
x_train = train_dataset.drop('income', axis=1)
x_test = test_dataset.drop('income', axis=1)


In [60]:
#Given the train dataset, we construct a data object for DiCE. 
# Since continuous and discrete features have different ways of perturbation, we need to specify the names of the continuous features. 
# DiCE also requires the name of the output variable that the ML model will predict.
# Step 1: dice_ml.Data
d = dice_ml.Data(dataframe=train_dataset, continuous_features=['age', 'hours_per_week'], outcome_name='income')

In [61]:
# Loading the ML model
#DiCE supports sklearn, tensorflow and pytorch models.
#The variable backend below indicates the implementation type of DiCE we want to use. Four backends are supported: sklearn, TensorFlow 1.x with backend=‘TF1’, Tensorflow 2.x with backend=‘TF2’, and PyTorch with backend=‘PYT’.

#Below we show use a trained classification model using sklearn.
numerical = ["age", "hours_per_week"]
categorical = x_train.columns.difference(numerical)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical)])

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', RandomForestClassifier())])
model = clf.fit(x_train, y_train)



Generating counterfactual examples using DiCE

In [62]:
#We now initialize the DiCE explainer, which needs a dataset and a model. DiCE provides local explanation for the model m and requires an query input whose outcome needs to be explained.
# Using sklearn backend
m = dice_ml.Model(model=model, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m, method="random")
#The method parameter specifies the explanation method. DiCE supports three methods for sklearn models: random sampling, genetic algorithm search, and kd-tree based generation.

In [63]:
#The next code snippet shows how to generate and visualize counterfactuals. 
# The first argument of the generate_counterfactuals method is the query instances on which counterfactuals are desired. 
# This can be a dataframe with one or more rows.

#Below we provide a sample input whose outcome is 0 (low-income) as per the ML model object m. 
# Given the query input, we can now generate counterfactual explanations to show perturbed inputs from the original input where the ML model outputs class 1 (high-income). 
# The last column shows the output of the classifier: income-output >=0.5 is class 1 and income-output<0.5 is class 0.

e1 = exp.generate_counterfactuals(x_test[0:1], total_CFs=2, desired_class="opposite")
e1.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00, 14.48it/s]

Query instance (original outcome : 0)



/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Masters,-,White-Collar,-,-,-,1
1,-,Government,-,Single,-,-,-,-,1


In [64]:
# Changing only age and education
e2 = exp.generate_counterfactuals(x_test[0:1],
                                  total_CFs=2,
                                  desired_class="opposite",
                                  features_to_vary=["education", "occupation"]
                                  )
e2.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00, 17.86it/s]

Query instance (original outcome : 0)



/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Doctorate,-,Professional,-,-,-,1
1,-,-,Masters,-,Professional,-,-,-,1


In [65]:
# Restricting age to be between [20,30] and Education to be either {'Doctorate', 'Prof-school'}.
e3 = exp.generate_counterfactuals(x_test[0:1],
                                  total_CFs=2,
                                  desired_class="opposite",
                                  permitted_range={'age': [20, 30], 'education': ['Doctorate', 'Prof-school']})
e3.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00, 18.16it/s]

Query instance (original outcome : 0)



/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,29,Private,HS-grad,Married,Blue-Collar,White,Female,38,0



Diverse Counterfactual set (new outcome: 1)


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,Government,Doctorate,-,-,-,-,-,1
1,-,-,Doctorate,-,-,-,Male,-,1


Generating feature attributions (local and global) using DiCE

In [66]:
# Local feature importance scores
query_instance = x_test[0:1]
imp = exp.local_feature_importance(query_instance, total_CFs=10)
print(imp.local_importance)

100%|██████████| 1/1 [00:00<00:00,  8.55it/s]

[{'education': 0.8, 'workclass': 0.4, 'occupation': 0.4, 'gender': 0.3, 'marital_status': 0.1, 'race': 0.0, 'age': 0.0, 'hours_per_week': 0.0}]


In [67]:
# Global feature importance scores
# A global importance score per feature can be estimated by aggregating the scores over individual inputs. The more the inputs, the better the estimate for global importance of a feature.

query_instances = x_test[0:20]
imp = exp.global_feature_importance(query_instances)
print(imp.summary_importance)

100%|██████████| 20/20 [00:02<00:00,  9.29it/s]

{'education': 0.59, 'occupation': 0.34, 'marital_status': 0.24, 'workclass': 0.215, 'age': 0.205, 'hours_per_week': 0.18, 'race': 0.14, 'gender': 0.14}


Advanced options to customize Counterfactual Explanations
https://interpret.ml/DiCE/notebooks/DiCE_with_advanced_options.html

In [68]:
from numpy.random import seed

# import DiCE
import dice_ml
from dice_ml.utils import helpers  # helper functions

# Tensorflow libraries
import tensorflow as tf

# supress deprecation warnings from TF
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:
dataset = helpers.load_adult_income_dataset()
dataset.head()
d = dice_ml.Data(dataframe=dataset, continuous_features=['age', 'hours_per_week'], outcome_name='income')


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/dice_ml/utils/helpers.py:79: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adult_data = adult_data.replace({'income': {'<=50K': 0, '>50K': 1}})


In [70]:
# Voy a hacer un gradient boosting classifier
from sklearn.ensemble import GradientBoostingClassifier

numerical = ["age", "hours_per_week"]
categorical = x_train.columns.difference(numerical)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical)])

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=42))])
mimodel = clf.fit(x_train, y_train)

In [71]:
# Generate diverse counterfactuals
# initiate DiCE
#We now initialize the DiCE explainer, which needs a dataset and a model. DiCE provides local explanation for the model m and requires an query input whose outcome needs to be explained.
# Using sklearn backend
m = dice_ml.Model(model=mimodel, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m, method="random")
#The method parameter specifies the explanation method. DiCE supports three methods for sklearn models: random sampling, genetic algorithm search, and kd-tree based generation.



# query instance in the form of a dictionary; keys: feature name, values: feature value
query_instance = {'age': 22,
                  'workclass': 'Private',
                  'education': 'HS-grad',
                  'marital_status': 'Single',
                  'occupation': 'Service',
                  'race': 'White',
                  'gender': 'Female',
                  'hours_per_week': 45}

query_df = pd.DataFrame([query_instance])
# Asegurarse de que las columnas del DataFrame coincidan con las de x_test
query_df = query_df[x_test.columns]

# Mostrar el DataFrame resultante
print(query_df)

   age workclass education marital_status occupation   race  gender   
0   22   Private   HS-grad         Single    Service  White  Female  \

   hours_per_week  
0              45  


In [72]:
# We now generate counterfactuals for this input.

dice_exp = exp.generate_counterfactuals(query_df, total_CFs=4, desired_class="opposite")
dice_exp.visualize_as_dataframe(show_only_changes=True)


100%|██████████| 1/1 [00:00<00:00, 21.15it/s]

Query instance (original outcome : 0)



/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,22,Private,HS-grad,Single,Service,White,Female,45,0



Diverse Counterfactual set (new outcome: 1)


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Doctorate,Married,-,-,-,-,1
1,-,-,Masters,Married,-,-,-,-,1
2,68,-,Masters,Married,-,-,-,-,1
3,-,-,Prof-school,Married,-,-,-,-,1


2. Changing feature weights

It may be the case that some features are harder to change than others (e.g., education level is harder to change than working hours per week). DiCE allows input of relative difficulty in changing a feature through specifying feature weights. A higher feature weight means that the feature is harder to change than others. For instance, one way is to use the mean absolute deviation from the median as a measure of relative difficulty of changing a continuous feature.

Median Absolute Deviation (MAD) of a continuous feature conveys the variability of the feature, and is more robust than standard deviation as is less affected by outliers and non-normality. The inverse of MAD would then imply the ease of varying the feature and is hence used as feature weights in our optimization to reflect the difficulty of changing a continuous feature. By default, DiCE computes this internally and divides the distance between continuous features by the MAD of the feature’s values in the training set. Let’s see what their values are by computing them below:

In [73]:
# get MAD
mads = d.get_mads(normalized=True)

# create feature weights
feature_weights = {}
for feature in mads:
    feature_weights[feature] = round(1/mads[feature], 2)
print(feature_weights)

#The above feature weights encode that changing age is approximately seven times more difficult than changing categorical variables, and changing hours_per_week is approximately three times more difficult than changing age. Of course, this may sound odd, since a person cannot change their age. In this case, what it’s reflecting is that there is a higher diversity in age values than hours-per-week values. 
# Below we show how to over-ride these weights to assign custom user-defined weights.

{'age': 7.3, 'hours_per_week': 24.5}


In [ ]:
help(dice_ml.Dice.generate_counterfactuals)
# Feature weights no está implementado como parámetro dice_ml.Dice (method="random") que es el valor por defecto, en la función generate_counterfactuals
#https://interpret.ml/DiCE/_modules/dice_ml/explainer_interfaces/dice_random.html

# así que su tutorial no funciona

Help on function generate_counterfactuals in module dice_ml.explainer_interfaces.explainer_base:

generate_counterfactuals(self, query_instances, total_CFs, desired_class='opposite', desired_range=None, permitted_range=None, features_to_vary='all', stopping_threshold=0.5, posthoc_sparsity_param=0.1, proximity_weight=0.2, sparsity_weight=0.2, diversity_weight=5.0, categorical_penalty=0.1, posthoc_sparsity_algorithm='linear', verbose=False, **kwargs)
    General method for generating counterfactuals.

    :param query_instances: Input point(s) for which counterfactuals are to be generated.
                            This can be a dataframe with one or more rows.
    :param total_CFs: Total number of counterfactuals required.
    :param desired_class: Desired counterfactual class - can take 0 or 1. Default value
                          is "opposite" to the outcome class of query_instance for binary classification.
    :param desired_range: For regression problems. Contains the outcome 

In [ ]:
# Now, let’s try to assign unit weights to the continuous features and see how it affects the counterfactual generation. 
# DiCE allows this through feature_weights parameter.

# assigning equal weights
feature_weights = {'age': 1, 'hours_per_week': 1}
# generate counterfactuals
dice_exp = exp.generate_counterfactuals(query_df, total_CFs=4, desired_class="opposite",feature_weights=feature_weights)
# visualize the resutls
dice_exp.visualize_as_dataframe(show_only_changes=True)

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: DiceRandom._generate_counterfactuals() got an unexpected keyword argument 'feature_weights'

In [ ]:
# Voy a probar con el method 'genetic' para generar CFs que es el que viene en el tutorial para generar Cfs sin acceso al modelo de ML, usando sólo los datos
exp2= dice_ml.Dice(d, m, method="genetic")

dice_exp2 = exp2.generate_counterfactuals(query_df, total_CFs=4, desired_class="opposite", feature_weights=feature_weights)
dice_exp2.visualize_as_dataframe(show_only_changes=True)


100%|██████████| 1/1 [00:00<00:00,  4.39it/s]

Query instance (original outcome : 0)



/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,22,Private,HS-grad,Single,Service,White,Female,45,0



Diverse Counterfactual set (new outcome: 1)


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Doctorate,Married,White-Collar,-,Male,-,1
0,-,-,Bachelors,Married,Blue-Collar,-,Male,-,1
0,-,-,Some-college,Married,White-Collar,-,Male,-,1
0,26,-,Bachelors,Married,White-Collar,-,-,-,1


Note that we transform continuous features and one-hot-encode categorical features to fall between 0 and 1 in order to handle relative scale of features. However, this also means that the relative ease of changing continuous features is higher than categorical features when the total number of continuous features are very less compared to the total number of categories of all categorical variables combined. This is reflected in the above table where continuous features (age and hours_per_week) have been varied to reach their extreme values (range of age: [17, 90]; range of hours_per_week: [1, 99]) for most of the counterfactuals. This is the reason why the distances are divided by a scaling factor. Deviation from the median provides a robust measure of the variability of a feature’s values, and thus dividing by the MAD allows us to capture the relative prevalence of observing the feature at a particular value (see our paper for more details).

In [81]:
# como no tiene sentido cambiar age, podría ponerle un peso muy alto o decir que no se manipule
# le voy a poner un peso muy alto

feature_weights = {'age': 100, 'hours_per_week': 1}
dice_exp2 = exp2.generate_counterfactuals(query_df, total_CFs=4, desired_class="opposite", feature_weights=feature_weights)
dice_exp2.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  3.73it/s]

Query instance (original outcome : 0)



/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,22,Private,HS-grad,Single,Service,White,Female,45,0



Diverse Counterfactual set (new outcome: 1)


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Some-college,Married,Professional,-,Male,-,1
0,-,-,Bachelors,Married,Blue-Collar,-,Male,-,1
0,-,-,Some-college,Married,White-Collar,-,Male,-,1
0,23,-,Bachelors,Married,Sales,-,-,-,1


3. Trading off between proximity and diversity goals
We acknowledge that not all counterfactual explanations may be feasible for a user. In general, counterfactuals closer to an individual’s profile will be more feasible. Diversity is also important to help an individual choose between multiple possible options. DiCE allows tunable parameters proximity_weight (default: 0.5) and diversity_weight (default: 1.0) to handle proximity and diversity respectively. Below, we increase the proximity weight and see how the counterfactuals change.

In [82]:
# change proximity_weight from default value of 0.5 to 1.5
dice_exp = exp.generate_counterfactuals(query_df, total_CFs=4, desired_class="opposite",
                                        proximity_weight=1.5, diversity_weight=1.0)
# visualize the resutls
dice_exp.visualize_as_dataframe(show_only_changes=True)


100%|██████████| 1/1 [00:00<00:00, 12.94it/s]

Query instance (original outcome : 0)



/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,22,Private,HS-grad,Single,Service,White,Female,45,0



Diverse Counterfactual set (new outcome: 1)


/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()
/Users/mariapereda/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Counterfactuals_trabajo/.venv/lib/python3.12/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Prof-school,Married,-,-,-,-,1
1,-,-,Doctorate,Married,-,-,Male,-,1
2,42,-,Doctorate,Married,-,-,-,-,1
3,-,-,Masters,Married,-,-,-,-,1
